# Options risk-neutral probability surfaces (London Strategic Edge)

Build implied-volatility and Breeden-Litzenberger risk-neutral density (RND) surfaces
from an LSE options chain, then read off risk-neutral probabilities -- e.g. "what does
the options market think are the odds this stock is up/down 10% by expiry?".

Requires `pip install "maroczy[lse,notebook]"` and an `LSE_API_KEY` (copy `.env.example`
to `.env` and fill it in -- see the main `quickstart.ipynb` for details).

## 0. Setup

In [1]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from scipy.interpolate import UnivariateSpline
from scipy.stats import norm

from maroczy.datafeed import LSEData

lse = LSEData()

## 1. Pull the options chain

LSE computes implied volatility and greeks itself from its own pricing models, so we
don't need to invert Black-Scholes to get IV -- we only re-derive the *risk-neutral
density* from the smile it hands us.

In [2]:
UNDERLYING = "AAPL"
MAX_DTE = 240

chain = lse.options_chain(UNDERLYING, max_dte=MAX_DTE)
print(f"{len(chain)} contracts returned for {UNDERLYING}")
chain.head()

2298 contracts returned for AAPL


,ticker,underlying,strike,expiry,contract_type,last_price,volume_today,premium_today,underlying_price,dte,iv,delta,gamma,theta,vega,rho,last_trade_at,updated_at
0,AAPL260702P00110000,AAPL,110.0,2026-07-02,put,0.01,4,18.0,293.07,1,3.3104,-0.0004,0.000017,-0.0213,0.0004,-0.0000,2026-07-01T14:16:50.700054Z,2026-07-01T14:17:20.667670Z
1,AAPL260702C00120000,AAPL,120.0,2026-07-02,call,174.47,1,17447.0,294.23,1,NaN,NaN,NaN,NaN,NaN,NaN,2026-07-01T14:46:50.189734Z,2026-07-01T14:47:18.449110Z
2,AAPL260702P00120000,AAPL,120.0,2026-07-02,put,0.01,1000,1000.0,295.36,8,1.9071,-0.0004,0.000018,-0.0080,0.0007,-0.0000,2026-06-24T19:04:34.201239Z,2026-06-24T19:04:50.743158Z
3,AAPL260702C00125000,AAPL,125.0,2026-07-02,call,169.36,3,50818.0,294.14,1,NaN,NaN,NaN,NaN,NaN,NaN,2026-07-01T17:34:34.071041Z,2026-07-01T17:34:43.936919Z
4,AAPL260702P00125000,AAPL,125.0,2026-07-02,put,0.03,12,34.0,281.13,6,2.1944,-0.0012,0.000052,-0.0271,0.0015,-0.0001,2026-06-26T19:22:43.864924Z,2026-06-26T19:23:09.614865Z


## 2. Resolve column names defensively

LSE's exact field names for the options-chain endpoint aren't pinned down in the public
docs, so we resolve common aliases here rather than hardcoding a guess -- print
`chain.columns.tolist()` and extend the candidate lists below if your response differs.

In [12]:
def find_column(df: pd.DataFrame, candidates: list[str], required: bool = True) -> str | None:
    for c in candidates:
        if c in df.columns:
            return c
    if required:
        raise KeyError(f"None of {candidates} found in columns: {df.columns.tolist()}")
    return None


print("chain columns:", chain.columns.tolist())

COL_STRIKE = find_column(chain, ["strike", "strike_price", "K"])
COL_EXPIRY = find_column(chain, ["expiry", "expiration", "expiration_date"])
COL_TYPE = find_column(chain, ["contract_type", "type", "option_type", "right"])
COL_IV = find_column(chain, ["iv", "implied_volatility", "implied_vol"])
COL_UNDERLYING_PX = find_column(chain, ["underlying_price", "spot", "underlying_last"], required=False)

chain[COL_EXPIRY] = pd.to_datetime(chain[COL_EXPIRY])
calls = chain[chain[COL_TYPE].astype(str).str.lower().str.startswith("c")].copy()
len(calls)


chain columns: ['ticker', 'underlying', 'strike', 'expiry', 'contract_type', 'last_price', 'volume_today', 'premium_today', 'underlying_price', 'dte', 'iv', 'delta', 'gamma', 'theta', 'vega', 'rho', 'last_trade_at', 'updated_at']


1258

## 3. Spot price & risk-free rate

Use the chain's own underlying price if it carries one, otherwise the latest daily
close; use the 1-year Treasury yield as a risk-free proxy (falls back to a flat default
if that tenor isn't available).

In [13]:
if COL_UNDERLYING_PX and chain[COL_UNDERLYING_PX].notna().any():
    S0 = float(chain[COL_UNDERLYING_PX].dropna().iloc[-1])
else:
    S0 = float(lse.candles(UNDERLYING, "1d", limit=1)["close"].iloc[-1])

try:
    y1 = lse.bond_yields("US1Y")
    y1_col = find_column(y1, ["value", "yield", "close"])
    r = float(y1[y1_col].iloc[-1]) / 100.0
except Exception:
    r = 0.045  # fallback risk-free rate if the US1Y tenor isn't available

print(f"S0 = {S0:.2f}   r = {r:.3%}")

S0 = 315.13   r = 3.935%


## 4. Black-Scholes pricing helper

In [14]:
def bs_call_price(S: float, K: np.ndarray, T: float, r: float, sigma: np.ndarray, q: float = 0.0) -> np.ndarray:
    T = np.maximum(T, 1e-6)
    sigma = np.maximum(sigma, 1e-4)
    d1 = (np.log(S / K) + (r - q + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)
    return S * np.exp(-q * T) * norm.cdf(d1) - K * np.exp(-r * T) * norm.cdf(d2)

## 5. Fit a smooth smile per expiry & compute the risk-neutral density

For each expiry: fit a smoothing spline to `IV(strike)` (raw market IVs are noisy;
naive interpolation makes the second derivative below blow up), evaluate it on a dense
strike grid, price synthetic calls with Black-Scholes, then apply the
**Breeden-Litzenberger** formula

$$ q(K) = e^{rT} \frac{\partial^2 C}{\partial K^2} $$

via finite differences. (Sanity-checked separately against the known analytic
lognormal density under flat Black-Scholes vol -- recovers it to ~1e-7 accuracy.)

In [15]:
def fit_expiry_curve(strikes: np.ndarray, ivs: np.ndarray, n_grid: int = 400, smoothing: float | None = None):
    order = np.argsort(strikes)
    strikes, ivs = np.asarray(strikes)[order], np.asarray(ivs)[order]
    strikes, unique_idx = np.unique(strikes, return_index=True)
    ivs = ivs[unique_idx]
    if len(strikes) < 5:
        return None
    s = smoothing if smoothing is not None else 0.02 * len(strikes)
    spline = UnivariateSpline(strikes, ivs, k=min(3, len(strikes) - 1), s=s)
    dense_K = np.linspace(strikes.min(), strikes.max(), n_grid)
    dense_iv = np.clip(spline(dense_K), 0.02, 3.0)
    return dense_K, dense_iv


def breeden_litzenberger(dense_K: np.ndarray, dense_iv: np.ndarray, S0: float, T: float, r: float) -> np.ndarray:
    C = bs_call_price(S0, dense_K, T, r, dense_iv)
    dK = dense_K[1] - dense_K[0]
    rnd = np.exp(r * T) * np.gradient(np.gradient(C, dK), dK)
    return np.clip(rnd, 0, None)


surfaces: dict[pd.Timestamp, dict] = {}
# `chain[COL_EXPIRY]` parses as tz-naive, so anchor "today" as tz-naive too
# (avoids "Cannot subtract tz-naive and tz-aware datetime-like objects").
today = pd.Timestamp.now().normalize()
for expiry, group in calls.groupby(COL_EXPIRY):
    T = max((expiry - today).days, 1) / 365.0
    fitted = fit_expiry_curve(group[COL_STRIKE].to_numpy(dtype=float), group[COL_IV].to_numpy(dtype=float))
    if fitted is None:
        continue
    dense_K, dense_iv = fitted
    rnd = breeden_litzenberger(dense_K, dense_iv, S0, T, r)
    surfaces[expiry] = {"T": T, "K": dense_K, "iv": dense_iv, "rnd": rnd}

print(f"Built RND/IV curves for {len(surfaces)} expiries")


Built RND/IV curves for 23 expiries


## 6. Plot the risk-neutral density surface

Strikes differ per expiry, so we re-interpolate each expiry's curve onto a common
**moneyness** grid (`K / S0`) to build a coherent 3D mesh.

In [16]:
moneyness_grid = np.linspace(0.5, 1.5, 120)
expiries_sorted = sorted(surfaces.keys())
Ts = np.array([surfaces[e]["T"] for e in expiries_sorted])

rnd_matrix = np.full((len(expiries_sorted), len(moneyness_grid)), np.nan)
iv_matrix = np.full((len(expiries_sorted), len(moneyness_grid)), np.nan)
for i, expiry in enumerate(expiries_sorted):
    data = surfaces[expiry]
    m = data["K"] / S0
    rnd_matrix[i] = np.interp(moneyness_grid, m, data["rnd"] * S0, left=np.nan, right=np.nan)
    iv_matrix[i] = np.interp(moneyness_grid, m, data["iv"], left=np.nan, right=np.nan)

fig_rnd = go.Figure(
    data=[
        go.Surface(
            x=moneyness_grid,
            y=Ts,
            z=rnd_matrix,
            colorscale="Viridis",
            colorbar={"title": "risk-neutral density"},
        )
    ]
)
fig_rnd.update_layout(
    title=f"{UNDERLYING}: risk-neutral density surface",
    scene={
        "xaxis_title": "moneyness (K / S0)",
        "yaxis_title": "time to expiry (yrs)",
        "zaxis_title": "density",
    },
    height=650,
)
fig_rnd.show()

## 7. Plot the implied volatility surface

In [17]:
fig_iv = go.Figure(
    data=[
        go.Surface(
            x=moneyness_grid,
            y=Ts,
            z=iv_matrix,
            colorscale="RdBu_r",
            colorbar={"title": "implied vol"},
        )
    ]
)
fig_iv.update_layout(
    title=f"{UNDERLYING}: implied volatility surface",
    scene={
        "xaxis_title": "moneyness (K / S0)",
        "yaxis_title": "time to expiry (yrs)",
        "zaxis_title": "IV",
    },
    height=650,
)
fig_iv.show()

## 8. Risk-neutral probabilities: "what does the market think?"

Integrate the density to get `P(S_T > threshold)` for each expiry -- a direct,
options-market-implied read on the odds of a move, no historical-vol assumption needed.

In [18]:
def risk_neutral_prob_above(K_grid: np.ndarray, rnd: np.ndarray, threshold: float) -> float:
    total = np.trapezoid(rnd, K_grid)
    if total <= 0:
        return np.nan
    mask = K_grid >= threshold
    return np.trapezoid(rnd[mask], K_grid[mask]) / total


thresholds = {"flat": S0, "+10%": S0 * 1.10, "-10%": S0 * 0.90, "+25%": S0 * 1.25}
rows = []
for expiry in expiries_sorted:
    data = surfaces[expiry]
    row = {"expiry": expiry.date(), "days_to_expiry": round(data["T"] * 365)}
    for label, thr in thresholds.items():
        row[f"P(S>{label})"] = risk_neutral_prob_above(data["K"], data["rnd"], thr)
    rows.append(row)

prob_table = pd.DataFrame(rows).set_index("expiry")
prob_table

,days_to_expiry,P(S>flat),P(S>+10%),P(S>-10%),P(S>+25%)
expiry,,,,,
2026-07-02,1,NaN,NaN,NaN,NaN
2026-07-06,1,NaN,NaN,NaN,NaN
2026-07-08,1,0.462215,0.000146,0.968650,0.000000
2026-07-10,1,NaN,NaN,NaN,NaN
2026-07-13,1,NaN,NaN,NaN,NaN
2026-07-15,1,0.486229,0.000013,0.971595,0.000000
2026-07-17,2,NaN,NaN,NaN,NaN
2026-07-20,5,NaN,NaN,NaN,NaN
2026-07-22,7,NaN,NaN,NaN,NaN


## Caveats

- LSE's own IV/greeks come from their pricing models; we only re-derive the *density*,
  not the IV itself -- garbage in (a noisy chain) still means a noisy surface out.
- The smoothing spline trades fidelity for numerical stability in the second
  derivative -- widen/narrow `smoothing` in `fit_expiry_curve` if the surface looks too
  flat or too jagged.
- Densities are truncated to the traded strike range (no tail extrapolation beyond the
  listed strikes), so `P(S > threshold)` for extreme thresholds understates the true
  probability mass sitting in the tails.
- Column names for the options-chain endpoint are resolved defensively
  (`find_column`) since exact field names weren't confirmed against a live response in
  this environment -- extend the alias lists above if your chain uses different names.